# Opsim Statistics per Band

- **author :** Sylvie Dagoret-Campagne
- **affiliation :** IJCLab/IN2P3/CNRS
- **creation date :** 2026-07-24
- **description :** Statistics and distributions of numerical columns per band (u,g,r,i,z,y)

## Requirements
- Install :
https://github.com/LSSTDESC/OpSimSummaryV2

In [ ]:
import opsimsummaryv2 as op
import os
import sys
import logging
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.time import Time

In [ ]:
# Not all figures are shown with widgets
#import ipywidgets
#import ipywidgets as widgets
#%matplotlib widget

print("matplotlib:", mpl.__version__)
print("backend:", mpl.get_backend())
#print("ipywidgets:", ipywidgets.__version__)

## 1) Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging implemented !")

## 2) Db Configuration

In [ ]:
# Notebook input

# -- Notebook tag ---------------------------------------------------------
NB_TAG = "OpsimStat_03"
DIR_DATA_IN = "/Users/dagoret/DATA/OpSim"

# -- Output figures --------------------------------------------------------
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

### List of opsim simulations

In [ ]:
# Opsim Rubin-LSST cadence simulations
list_opsims = ["baseline_v5.3.0_10yrs.db","faster_templates_v5.3.0_10yrs.db","ddf_one_less_v5.3.2_10yrs.db","roll_mash_v5.3.0_10yrs.db",
               "roll_u5_v5.3.0_10yrs.db","ddf_sd_v5.3.0_10yrs.db","desi_3040_v5.3.0_10yrs.db","baseline_v5.3.0_11yrs.db"]

In [ ]:
index_sim = 0
filename_in = list_opsims[index_sim]
opsim_path = os.path.join(DIR_DATA_IN, filename_in)

In [ ]:
log.info("Selected Opsim cadence simulation : %s", filename_in)
tagname = re.match(r"(.+)\.db$", filename_in)
if tagname:
    tagname = tagname.group(1)
    log.info(f"tag = {tagname}")

In [ ]:
log.info("Configuration of opsim %s done!", opsim_path)

## 3) Read Opsim Db

In [ ]:
sql_engine = op.OpSimSurvey._get_sql_engine(opsim_path)

In [ ]:
df = op.OpSimSurvey._get_df_from_sql(sql_engine)

In [ ]:
log.info(f"columns : {list(df.columns)}")

## 4) Analyse Setup

In [ ]:
BANDS_ORDER = ["u", "g", "r", "i", "z", "y"]

In [ ]:
df["filter"] = df["filter"].str.extract(r"([a-zA-Z]+)")

In [ ]:
all_df = [df[df["filter"] == b] for b in BANDS_ORDER]

## 5) Plots

In [ ]:
# Standard Rubin/LSST filter colors
FILTER_COLORS = {
    "u": "#56b4e9",
    "g": "#008060",
    "r": "#ff4000",
    "i": "#850000",
    "z": "#6600cc",
    "y": "#000000",
}
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]

### 5.1) Helpers

In [ ]:
# -- Helper to clean data from NaN and inf ---------------------------------
def clean_data(data):
    """Remove NaN and infinite values from a numpy array or pandas Series."""
    if isinstance(data, pd.Series):
        data = data.to_numpy()
    return data[(~np.isnan(data)) & np.isfinite(data)]


In [ ]:
# -- savefig: PDF + PNG ------------------------------------------------
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    #fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
def compute_statistics(data):
    """
    Compute statistics for a given dataset: mean, median, sigma_mad, and RMS.
    Returns a dictionary with values truncated to 3 significant figures.
    """
    # Clean data: remove NaN and infinite values (with proper parentheses)
    clean_data = data[(~np.isnan(data)) & np.isfinite(data)]
    
    if len(clean_data) == 0:
        log.warning("All data are NaN or infinite, cannot compute statistics")
        return {
            'mean': np.nan,
            'median': np.nan,
            'sigma_mad': np.nan,
            'rms': np.nan
        }
    
    mean_val = np.mean(clean_data)
    median_val = np.median(clean_data)
    
    # Sigma MAD: 1.4826 * median absolute deviation
    mad = np.median(np.abs(clean_data - median_val))
    sigma_mad_val = 1.4826 * mad
    
    # RMS: root mean square
    rms_val = np.sqrt(np.mean(clean_data**2))
    
    # Format to 3 significant figures
    def format_to_3sf(val):
        if np.isnan(val):
            return "nan"
        return f"{val:.3g}"
    
    return {
        'mean': format_to_3sf(mean_val),
        'median': format_to_3sf(median_val),
        'sigma_mad': format_to_3sf(sigma_mad_val),
        'rms': format_to_3sf(rms_val)
    }

In [ ]:
def plot_distribution_per_band(df, column, filter_column="filter", simuname=tagname, figsize=(14, 6)):
    """
    Plot the distribution of a numerical column per band (u,g,r,i,z,y) in a 2x3 subplot grid.
    Each subplot shows the histogram for one band with statistics in a textbox.
    """
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    axes = axes.ravel()  # Flatten to 1D array
    
    for idx, band in enumerate(FILTER_ORDER):
        ax = axes[idx]
        
        # Filter data for this band
        mask = df[filter_column] == band
        band_data = df[column][mask]
        
        # Clean data: remove NaN and infinite values (with proper parentheses for operator precedence)
        clean_data = band_data[(~np.isnan(band_data)) & np.isfinite(band_data)]
        
        if len(clean_data) == 0:
            ax.set_title(f"{band}: No valid data")
            ax.axis('off')
            log.warning(f"No valid data for {column} in band {band} (all NaN/inf)")
            continue
        
        # Plot histogram with band color
        n, bins, patches = ax.hist(
            clean_data,
            bins=50,
            density=True,
            color=FILTER_COLORS[band],
            alpha=0.7,
            edgecolor='black',
            linewidth=0.5
        )
        
        # Compute statistics
        stats = compute_statistics(clean_data)
        
        # Create text box with statistics
        stats_text = (
            f"{band} band:\n"
            f"mean:   {stats['mean']}\n"
            f"median: {stats['median']}\n"
            f"sigma_MAD: {stats['sigma_mad']}\n"
            f"RMS:    {stats['rms']}"
        )
        
        # Add text box in the upper left corner
        ax.text(
            0.65,
            0.98,
            stats_text,
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'),
            fontsize=9,
            fontfamily='monospace'
        )
        
        ax.set_xlabel(column)
        ax.set_ylabel('Density')
        ax.set_title(f"{band} band distribution")
        ax.grid(True, alpha=0.3, linestyle='--')
    
    # Adjust layout and main title
    fig.suptitle(f"{column} distribution by band ({simuname})", fontsize=16, y=1.0)
    fig.tight_layout()
    
    return fig, axes

### 5.2) Numerical Columns: Distribution per band

In [ ]:
# Columns not plotted: the MJD itself, and non-numeric / categorical columns
SKIP_COLUMNS = {"observationStartMJD", "band", "filter"}

for column in df.columns:
    if column in SKIP_COLUMNS:
        continue
    if not pd.api.types.is_numeric_dtype(df[column]):
        continue
    
    # Check if column has any non-NaN values
    if df[column].isna().all():
        log.warning(f"Column {column} contains only NaN values, skipping")
        continue
    
    # Create the distribution plot
    try:
        fig, axes = plot_distribution_per_band(df, column, simuname=tagname)
        
        # Save the figure
        figname = f"{tagname}_{column}_per_band"
        savefig(fig, figname, dpi=150)
        
        plt.show()
    except Exception as e:
        log.error(f"Failed to plot {column}: {e}")
        continue

In [ ]:
# Add raDeg/decDeg columns for completeness


def find_ra_dec_columns(df):
    """Find the RA/Dec column names in *df*, matching e.g. fieldRA/fieldDec,
    ra/dec, field_ra/field_dec (case-insensitive)."""
    ra_cols = [c for c in df.columns if re.search(r"(?i)(^|_)ra$", c)]
    dec_cols = [c for c in df.columns if re.search(r"(?i)(^|_)dec$", c)]
    if not ra_cols or not dec_cols:
        return None, None
    return ra_cols[0], dec_cols[0]

def add_radeg_decdeg(df, ra_col=None, dec_col=None):
    """Add raDeg (0-360 deg) and decDeg (-90 to +90 deg) columns to *df*,
    auto-converting from radians if needed."""
    if ra_col is None or dec_col is None:
        ra_col, dec_col = find_ra_dec_columns(df)
    if ra_col is None or dec_col is None:
        return df
    
    ra = df[ra_col].to_numpy(dtype=float)
    dec = df[dec_col].to_numpy(dtype=float)
    
    # Handle NaN values explicitly
    ra_has_nan = np.any(np.isnan(ra))
    dec_has_nan = np.any(np.isnan(dec))
    if ra_has_nan:
        log.warning(f"Found {np.sum(np.isnan(ra))} NaN values in RA column {ra_col}")
    if dec_has_nan:
        log.warning(f"Found {np.sum(np.isnan(dec))} NaN values in Dec column {dec_col}")
    
    ra_deg = np.degrees(ra) if np.nanmax(np.abs(ra)) <= (2 * np.pi + 0.01) else ra.copy()
    dec_deg = np.degrees(dec) if np.nanmax(np.abs(dec)) <= (np.pi / 2 + 0.01) else dec.copy()
    
    df["raDeg"] = np.mod(ra_deg, 360.0)
    df["decDeg"] = dec_deg
    
    log.info("raDeg/decDeg added from columns (%s, %s)", ra_col, dec_col)
    return df

df = add_radeg_decdeg(df)

In [ ]:
# Also plot raDeg and decDeg distributions per band
for column in ["raDeg", "decDeg"]:
    if column in df.columns:
        try:
            fig, axes = plot_distribution_per_band(df, column, simuname=tagname)
            figname = f"{tagname}_{column}_per_band"
            savefig(fig, figname, dpi=150)
            plt.show()
        except Exception as e:
            log.error(f"Failed to plot {column}: {e}")
            continue